# Quantization-aware training of ePC (QAT ePC)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thebuckleylab/jpc/blob/main/examples/qat_epc.ipynb)

This notebook demonstrates **quantization-aware training (QAT)** of an error-reparameterised PC network ([ePC](https://arxiv.org/abs/2505.20137); [Goemaere et al., 2025](https://arxiv.org/abs/2505.20137)) â€” see [`epc.ipynb`](./epc.ipynb) for the unquantized baseline.

We use the standard QAT recipe of **fake quantization** with a **straight-through estimator (STE)**: weights and activations are rounded to a small finite set of representable values on the forward pass, but gradients flow through the rounding step as the identity. The model parameters stay in full precision; only their *effective* values used in the matmul are quantized.

Two quantization modes are supported:
- **W4A8**: 4-bit weights, 8-bit activations (inference errors, inter-layer signals).
- **W4A4**: 4-bit weights, 4-bit activations.

and two number formats:
- **Posit** (primary): Posit\<4,2\> for weights and Posit\<8,2\> for activations, mirroring the `Posit4.jl` LUT-based reference and the `softposit` Python package for 8-bit.
- **FP** (ablation): FP4 `E2M1` (MXFP4-style) and FP8 `E4M3`.

Rather than depending on an external posit library at forward-pass time, we pre-compute the full set of representable values for each format once, then quantize by nearest-neighbour search against that grid. This keeps the forward pass pure JAX and `jit`-friendly, and makes Posit\<n,es\> and FP\<e,m\> drop-in interchangeable.

In [ ]:
%%capture
!pip install torch==2.3.1
!pip install torchvision==0.18.1

In [11]:
import math
from typing import Optional

import jpc

import numpy as np
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import equinox.nn as nn
import optax

import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import warnings
warnings.simplefilter('ignore')

## Hyperparameters

In [12]:
WIDTH = 128
DEPTH = 10
ACT_FN = "relu"

STATE_LR = 5e-1   # error learning rate (inference)
PARAM_LR = 1e-3   # parameter learning rate
BATCH_SIZE = 64
TEST_EVERY = 100
N_TRAIN_ITERS = 300

## Quantization grids

Each format is defined by a sorted 1D array of its representable finite values. At forward time, a tensor is quantized by:

1. computing a per-tensor **absmax scale** so the largest magnitude maps to the largest representable level,
2. snapping every scaled element to the **nearest level**,
3. rescaling back.

Special encodings (NaR for posits, NaN for FP) are dropped from the grid â€” they should never appear in a well-scaled tensor and would break nearest-neighbour search.

In [13]:
def enumerate_posit(n: int, es: int) -> np.ndarray:
    """All finite values representable by Posit<n,es>, sorted, NaR excluded.

    Matches the Posit<4,es> tables in ``Posit4.jl`` (useed = 2**(2**es)).
    """
    useed = 2 ** (2 ** es)
    nar = 1 << (n - 1)
    mask = (1 << n) - 1
    out = set()
    for bits in range(1 << n):
        if bits == nar:
            continue
        if bits == 0:
            out.add(0.0); continue
        s = (bits >> (n - 1)) & 1
        mag = bits if s == 0 else ((~bits) + 1) & mask
        rb = mag & ((1 << (n - 1)) - 1)
        pos = n - 2
        r_sign = (rb >> pos) & 1
        m = 0
        while pos >= 0 and ((rb >> pos) & 1) == r_sign:
            m += 1; pos -= 1
        k = (m - 1) if r_sign == 1 else -m
        if pos >= 0:
            pos -= 1  # terminating bit
        exp_avail = min(es, pos + 1)
        if exp_avail > 0:
            e = (rb >> (pos - exp_avail + 1)) & ((1 << exp_avail) - 1)
            pos -= exp_avail
        else:
            e = 0
        fb = pos + 1
        f = ((rb & ((1 << fb) - 1)) / (1 << fb)) if fb > 0 else 0.0
        v = (useed ** k) * (2 ** e) * (1 + f)
        out.add(-v if s else v)
    return np.array(sorted(out), dtype=np.float32)


def enumerate_fp(ebits: int, mbits: int, bias: int, has_nan: bool = True) -> np.ndarray:
    """All finite values of an IEEE-style minifloat, sorted, NaN/Inf excluded.

    ``has_nan=True`` mirrors FP8 E4M3 (no Inf; NaN at Emax, mantissa=all-ones).
    ``has_nan=False`` mirrors MXFP4 E2M1, where every codepoint is a finite number.
    """
    total = 1 + ebits + mbits
    emax = (1 << ebits) - 1
    mmax = (1 << mbits) - 1
    out = set()
    for bits in range(1 << total):
        s = (bits >> (total - 1)) & 1
        e = (bits >> mbits) & emax
        m = bits & mmax
        if e == 0:
            v = (m / (1 << mbits)) * (2 ** (1 - bias))
        elif has_nan and e == emax and m == mmax:
            continue  # NaN slot
        else:
            v = (1 + m / (1 << mbits)) * (2 ** (e - bias))
        out.add(-v if s else v)
    return np.array(sorted(out), dtype=np.float32)


# Precomputed level grids. Kept as JAX arrays for fast nearest-neighbour search.
LEVELS = {
    "posit4_0":  jnp.asarray(enumerate_posit(4, 0)),
    "posit4_1":  jnp.asarray(enumerate_posit(4, 1)),
    "posit4_2":  jnp.asarray(enumerate_posit(4, 2)),
    "posit8_2":  jnp.asarray(enumerate_posit(8, 2)),
    "fp4_e2m1":  jnp.asarray(enumerate_fp(ebits=2, mbits=1, bias=1, has_nan=False)),
    "fp8_e4m3":  jnp.asarray(enumerate_fp(ebits=4, mbits=3, bias=7, has_nan=True)),
}

for name, lv in LEVELS.items():
    pos = lv[lv > 0]
    print(f"{name:10s}  {lv.size:3d} levels   "
          f"min|x|={float(pos.min()):.4g}   max|x|={float(pos.max()):.4g}")

posit4_0     15 levels   min|x|=0.25   max|x|=4
posit4_1     15 levels   min|x|=0.0625   max|x|=16
posit4_2     15 levels   min|x|=0.003906   max|x|=256
posit8_2    255 levels   min|x|=5.96e-08   max|x|=1.678e+07
fp4_e2m1     15 levels   min|x|=0.5   max|x|=6
fp8_e4m3    253 levels   min|x|=0.001953   max|x|=448


## Fake quantization and `QuantLinear`

`fake_quant` does per-tensor RMS scaling, nearest-level rounding, rescaling, and wraps the result in a **straight-through estimator** (`x + stop_gradient(x_q - x)`). We use RMS rather than absmax because posit precision peaks at $|x|=1$ — absmax scaling would squash typical values into the posit tails and waste most of the grid.

`QuantLinear` is a drop-in replacement for `eqx.nn.Linear` that quantizes its input activations with the chosen `a_format` and its weights with the chosen `w_format`. Because `QuantLinear` subclasses `eqx.nn.Linear`, it exposes a `.weight` attribute — matching what JPC's internals inspect when computing PC energies and gradients.

In [14]:
def fake_quant(x: jax.Array, levels: jax.Array) -> jax.Array:
    """Quantize ``x`` to the grid ``levels`` with RMS scaling + STE.

    - Per-tensor scale = RMS(x) so typical magnitudes map to |x|~1, where both
      posit and FP grids have their densest precision.
    - Nearest-neighbour rounding against the grid.
    - Straight-through gradient: ``grad(x_q) == grad(x)``.
    """
    scale = jnp.sqrt(jnp.mean(x * x) + 1e-12)
    x_scaled = x / scale
    diffs = jnp.abs(x_scaled[..., None] - levels)
    idx = jnp.argmin(diffs, axis=-1)
    x_q = levels[idx] * scale
    return x + jax.lax.stop_gradient(x_q - x)


class QuantLinear(nn.Linear):
    """``eqx.nn.Linear`` with fake-quantized weights and input activations."""

    w_format: str = eqx.field(static=True)
    a_format: str = eqx.field(static=True)

    def __init__(
        self,
        in_features: int,
        out_features: int,
        *,
        use_bias: bool = True,
        key,
        w_format: str,
        a_format: str,
    ):
        super().__init__(in_features, out_features, use_bias=use_bias, key=key)
        self.w_format = w_format
        self.a_format = a_format

    def __call__(self, x, *, key=None):
        x_q = fake_quant(x, LEVELS[self.a_format])
        W_q = fake_quant(self.weight, LEVELS[self.w_format])
        y = W_q @ x_q
        if self.bias is not None:
            y = y + self.bias
        return y


def make_quant_mlp(
    key,
    input_dim: int,
    width: int,
    depth: int,
    output_dim: int,
    act_fn: str,
    w_format: str,
    a_format: str,
    use_bias: bool = True,
):
    """Mirror of ``jpc.make_mlp`` that uses ``QuantLinear`` instead of ``nn.Linear``."""
    subkeys = jr.split(key, depth)
    act_lookup = {"relu": jax.nn.relu, "tanh": jnp.tanh, "gelu": jax.nn.gelu}
    layers = []
    for i in range(depth):
        act = nn.Identity() if i == 0 else nn.Lambda(act_lookup[act_fn])
        _in = input_dim if i == 0 else width
        _out = output_dim if (i + 1) == depth else width
        linear = QuantLinear(
            _in, _out,
            use_bias=use_bias,
            key=subkeys[i],
            w_format=w_format,
            a_format=a_format,
        )
        layers.append(nn.Sequential([act, linear]))
    return layers


### Sanity check

Round-trip a random tensor through each format to confirm the quantizer snaps to the grid and that the STE leaves gradients untouched.

In [15]:
_rng = jr.PRNGKey(0)
_x = jr.normal(_rng, (1024,))
for fmt in LEVELS:
    _xq = fake_quant(_x, LEVELS[fmt])
    _err = float(jnp.sqrt(jnp.mean((_xq - _x) ** 2)))
    print(f"{fmt:10s}  RMSE(x, fake_quant(x)) = {_err:.4f}")

# STE check: gradient of a quantized identity should be 1.
_g = jax.grad(lambda x: fake_quant(x, LEVELS["posit4_2"]).sum())(_x)
print(f"STE grad mean = {float(_g.mean()):.6f}   (expected 1.0)")

posit4_0    RMSE(x, fake_quant(x)) = 0.1300
posit4_1    RMSE(x, fake_quant(x)) = 0.1929
posit4_2    RMSE(x, fake_quant(x)) = 0.2657
posit8_2    RMSE(x, fake_quant(x)) = 0.0260
fp4_e2m1    RMSE(x, fake_quant(x)) = 0.1509
fp8_e4m3    RMSE(x, fake_quant(x)) = 0.0260
STE grad mean = 1.000000   (expected 1.0)


## Dataset

Standard MNIST loader, identical to [`epc.ipynb`](./epc.ipynb).

In [16]:
def get_mnist_loaders(batch_size):
    train_data = MNIST(train=True, normalise=True)
    test_data = MNIST(train=False, normalise=True)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, drop_last=True)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=True, drop_last=True)
    return train_loader, test_loader


class MNIST(datasets.MNIST):
    def __init__(self, train, normalise=True, save_dir="data"):
        if normalise:
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
            ])
        else:
            transform = transforms.Compose([transforms.ToTensor()])
        super().__init__(save_dir, download=True, train=train, transform=transform)

    def __getitem__(self, index):
        img, label = super().__getitem__(index)
        return torch.flatten(img), one_hot(label)


def one_hot(labels, n_classes=10):
    return torch.eye(n_classes)[labels]

## Train and test

The training loop is the standard ePC recipe: initialize zero errors, relax them for `len(model)` steps with `jpc.update_epc_errors`, then make a single parameter update with `jpc.update_epc_params`. The only change is that the model is built by `make_quant_mlp` rather than `jpc.make_mlp` â€” every `W_q @ x_q` matmul inside the error and parameter energies is now quantized on the forward pass, while backprop uses the STE pass-through.

In [17]:
def evaluate(model, test_loader):
    avg_test_acc = 0.0
    for _, (img_batch, label_batch) in enumerate(test_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()
        _, test_acc = jpc.test_discriminative_pc(
            model=model, input=img_batch, output=label_batch
        )
        avg_test_acc += test_acc
    return avg_test_acc / len(test_loader)


def train(
    width,
    depth,
    act_fn,
    state_lr,
    param_lr,
    batch_size,
    test_every,
    n_train_iters,
    w_format: str,
    a_format: str,
    seed: int = 5482,
):
    key = jr.PRNGKey(seed)
    model = make_quant_mlp(
        key,
        input_dim=784,
        width=width,
        depth=depth,
        output_dim=10,
        act_fn=act_fn,
        w_format=w_format,
        a_format=a_format,
        use_bias=True,
    )
    layer_sizes = [784] + [width] * (depth - 1) + [10]

    error_optim = optax.sgd(state_lr)
    param_optim = optax.adam(param_lr)
    param_opt_state = param_optim.init((eqx.filter(model, eqx.is_array), None))

    train_loader, test_loader = get_mnist_loaders(batch_size)

    for it, (img_batch, label_batch) in enumerate(train_loader):
        img_batch, label_batch = img_batch.numpy(), label_batch.numpy()

        # feedforward pass for training-loss logging only
        activities = jpc.init_activities_with_ffwd(model=model, input=img_batch)
        train_loss = jpc.mse_loss(activities[-1], label_batch)

        # ePC: zero errors, relax, update
        errors = jpc.init_epc_errors(layer_sizes=layer_sizes, batch_size=batch_size)
        error_opt_state = error_optim.init(errors)

        for _ in range(len(model)):
            error_update_result = jpc.update_epc_errors(
                params=(model, None),
                errors=errors,
                optim=error_optim,
                opt_state=error_opt_state,
                output=label_batch,
                input=img_batch,
            )
            errors = error_update_result["errors"]
            error_opt_state = error_update_result["opt_state"]

        param_update_result = jpc.update_epc_params(
            params=(model, None),
            errors=errors,
            optim=param_optim,
            opt_state=param_opt_state,
            output=label_batch,
            input=img_batch,
        )
        model = param_update_result["model"]
        param_opt_state = param_update_result["opt_state"]

        if np.isinf(train_loss) or np.isnan(train_loss):
            print(f"Stopping training because of divergence, train loss={train_loss}")
            break

        if ((it + 1) % test_every) == 0:
            avg_test_acc = evaluate(model=model, test_loader=test_loader)
            print(
                f"[{w_format}/{a_format}] iter {it + 1}, "
                f"train loss={float(train_loss):.4f}, "
                f"test acc={float(avg_test_acc):.2f}%"
            )
            if (it + 1) >= n_train_iters:
                break

    return model

## Run — W4A8 posits (primary target)

4-bit **Posit\<4,2\>** weights and 8-bit **Posit\<8,2\>** activations. The Posit\<4,2\> grid is the same as the one tabulated in the accompanying `Posit4.jl` reference.

In [18]:
_ = train(
    width=WIDTH,
    depth=DEPTH,
    act_fn=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    w_format="posit4_2",
    a_format="posit8_2",
)

[posit4_2/posit8_2] iter 100, train loss=0.2199, test acc=78.45%
[posit4_2/posit8_2] iter 200, train loss=0.1613, test acc=90.06%
[posit4_2/posit8_2] iter 300, train loss=0.0307, test acc=92.40%


## Run — W4A4 posits

Pushing activations down to Posit\<4,2\> as well. This is the most aggressive configuration — expect a larger accuracy gap vs. the unquantized baseline in [`epc.ipynb`](./epc.ipynb).

In [19]:
_ = train(
    width=WIDTH,
    depth=DEPTH,
    act_fn=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    w_format="posit4_2",
    a_format="posit4_2",
)

[posit4_2/posit4_2] iter 100, train loss=0.3431, test acc=41.27%
[posit4_2/posit4_2] iter 200, train loss=0.2176, test acc=66.55%
[posit4_2/posit4_2] iter 300, train loss=0.1251, test acc=77.59%


## Ablation — FP4/FP8

Swapping in MXFP4-style `E2M1` weights and `E4M3` FP8 activations. Same training recipe, same number of mantissa+exponent bits per value; only the distribution of representable magnitudes differs. Running both W4A8 and W4A4 variants for direct comparison with the posit rows above.

In [20]:
_ = train(
    width=WIDTH,
    depth=DEPTH,
    act_fn=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    w_format="fp4_e2m1",
    a_format="fp8_e4m3",
)

_ = train(
    width=WIDTH,
    depth=DEPTH,
    act_fn=ACT_FN,
    state_lr=STATE_LR,
    param_lr=PARAM_LR,
    batch_size=BATCH_SIZE,
    test_every=TEST_EVERY,
    n_train_iters=N_TRAIN_ITERS,
    w_format="fp4_e2m1",
    a_format="fp4_e2m1",
)

[fp4_e2m1/fp8_e4m3] iter 100, train loss=0.2251, test acc=72.08%
[fp4_e2m1/fp8_e4m3] iter 200, train loss=0.0864, test acc=88.42%
[fp4_e2m1/fp8_e4m3] iter 300, train loss=0.1044, test acc=91.22%
[fp4_e2m1/fp4_e2m1] iter 100, train loss=0.1876, test acc=70.19%
[fp4_e2m1/fp4_e2m1] iter 200, train loss=0.1419, test acc=89.24%
[fp4_e2m1/fp4_e2m1] iter 300, train loss=0.0583, test acc=90.96%


## Ablation — Posit ES sweep

FP4 `E2M1` matched (or slightly beat) Posit\<4,2\> above, despite the posit having both a wider dynamic range and denser precision around $|x|=1$. A plausible read: with only 15 finite codepoints, Posit\<4,2\> spends many of them on magnitudes MNIST's RMS-scaled tensors never reach ($|x| \in \{16, 256\}$ and their reciprocals), leaving the "useful" band near 1 coarser than FP4's. Dropping `ES` trades dynamic range for denser spacing in that band — the opposite of what large-model 4-bit inference wants (where emergent outliers dominate), but potentially a better fit for a small, well-conditioned MNIST MLP.

Grid comparison (positive side, 7 non-zero levels each):

| Format | min\|x\| | max\|x\| | useed |
| :-- | --: | --: | --: |
| Posit\<4,0\> | 1/4 | 4 | 2 |
| Posit\<4,1\> | 1/16 | 16 | 4 |
| Posit\<4,2\> | 1/256 | 256 | 16 |

Running both W4A8 (Posit\<8,2\> activations held fixed) and W4A4 (same 4-bit format on both sides) for each `ES`, so the W4A4 variant also stresses activations that are close to $|x|=1$ after RMS scaling.

In [21]:
for es in (0, 1):
    w_fmt = f"posit4_{es}"

    # W4A8 variant: 4-bit posit weights, Posit<8,2> activations.
    _ = train(
        width=WIDTH,
        depth=DEPTH,
        act_fn=ACT_FN,
        state_lr=STATE_LR,
        param_lr=PARAM_LR,
        batch_size=BATCH_SIZE,
        test_every=TEST_EVERY,
        n_train_iters=N_TRAIN_ITERS,
        w_format=w_fmt,
        a_format="posit8_2",
    )

    # W4A4 variant: same 4-bit posit for weights and activations.
    _ = train(
        width=WIDTH,
        depth=DEPTH,
        act_fn=ACT_FN,
        state_lr=STATE_LR,
        param_lr=PARAM_LR,
        batch_size=BATCH_SIZE,
        test_every=TEST_EVERY,
        n_train_iters=N_TRAIN_ITERS,
        w_format=w_fmt,
        a_format=w_fmt,
    )

[posit4_0/posit8_2] iter 100, train loss=0.2000, test acc=68.95%
[posit4_0/posit8_2] iter 200, train loss=0.0674, test acc=90.34%
[posit4_0/posit8_2] iter 300, train loss=0.0791, test acc=92.19%
[posit4_0/posit4_0] iter 100, train loss=0.2851, test acc=52.54%
[posit4_0/posit4_0] iter 200, train loss=0.1867, test acc=74.35%
[posit4_0/posit4_0] iter 300, train loss=0.1032, test acc=86.46%
[posit4_1/posit8_2] iter 100, train loss=0.2056, test acc=74.75%
[posit4_1/posit8_2] iter 200, train loss=0.0911, test acc=89.18%
[posit4_1/posit8_2] iter 300, train loss=0.0514, test acc=93.05%
[posit4_1/posit4_1] iter 100, train loss=0.2546, test acc=69.70%
[posit4_1/posit4_1] iter 200, train loss=0.1224, test acc=82.41%
[posit4_1/posit4_1] iter 300, train loss=0.1585, test acc=89.92%


## Notes and extensions

- **What "QAT" means here.** Master weights and optimizer state stay in float32; only the forward pass matmuls and the activations that feed the next layer see quantized values. The STE means the ePC error/parameter gradients are computed as if the quantizer were the identity, so every JPC update function (`update_epc_errors`, `update_epc_params`) works unchanged.
- **Scaling.** We use per-tensor absmax scaling. For weights that is static per forward call; for activations it is dynamic (computed from the current batch). A common next step is per-output-channel weight scales, which typically recovers most of the remaining W4 accuracy gap.
- **Posit\<8,2\> vs. softposit.** For self-containedness the Posit\<8,2\> grid here is generated by the same pure-Python decoder used for Posit\<4,2\> â€” cross-check against `softposit.posit8(...)` if you have it installed; every one of the 255 finite values should match.
- **Swapping ePC for incremental/bidirectional variants.** The `make_quant_mlp` + `QuantLinear` pair is independent of the PC algorithm. Pointing [`iepc.ipynb`](./iepc.ipynb), [`bepc.ipynb`](./bepc.ipynb), or [`biepc.ipynb`](./biepc.ipynb) at `make_quant_mlp` instead of `jpc.make_mlp` gives the corresponding QAT variant with no other changes.